# Superlet Burst Analysis Pipeline

---

### **Author:** Juliette Leclerc  

### Description:
This custom Python pipeline was developed from scratch to perform robust downstream statistical analysis on oscillatory bursts extracted from *in vivo* electrophysiology data by `superlet_burst_detection.ipynb`.
### Key Features & Architecture:

1. **Automated Data Aggregation:** Dynamically loads, concatenates, and structures multi-subject, multi-session burst datasets (`.csv`/`.pkl`) into comprehensive `pandas` DataFrames.
2. **Advanced Statistical Modeling:** Implements **Generalized Estimating Equations (GEE)** via `statsmodels` to rigorously evaluate condition effects (e.g., Pre vs. Post-stimulation). This approach accounts for repeated measures and intra-subject correlations inherent to burst detection, bypassing the flawed independence assumptions of standard ANOVAs.
3. **Waveform & Feature Wrangling:** Handles high-dimensional data, extracting and visualizing both scalar features (duration, frequency span, peak amplitude) and raw 2D burst waveforms.
4. **Publication-Ready Visualization:** Integrates `seaborn` and `matplotlib` to generate clear, statistically-backed plots comparing burst dynamics across experimental conditions.

---

In [1]:
#Imports

#tkinter
import tkinter as tk
from tkinter import filedialog

#system
import os
import json
import glob
import re
import warnings
import inspect

import quantities as pq
import pickle
from time import time

import numpy as np
import pandas as pd
from tqdm import tqdm

#Plots and exports
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator

import statsmodels.api as sm
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Gaussian, Poisson
from statsmodels.genmod.cov_struct import Exchangeable

from scipy.io import savemat

## I. Loading files and related informations

In [ ]:
def file_choice():
    """ Opens a folder dialog repeatedly to let the user choose folders.
        Returns a list of all '_NaN_neo.mat' files from all selected folders.
    
    Output:
        - files : list of str, full paths to the selected files
    """

    all_files = []

    while True:
        # Create a hidden Tkinter root window (required for dialogs)
        root = tk.Tk()
        root.attributes("-topmost", True)
        root.update()
        root.withdraw()

        # Ask the user to select a folder
        folder_path = filedialog.askdirectory(title="Indicate where to find neo files (Cancel to finish)")

        # Destroy the hidden root window
        root.destroy()

        # If user cancels, break the loop
        if not folder_path:
            break

        # Add all matching files from this folder
        files = glob.glob(f"{folder_path}/*_neo_NaN.mat")
        all_files.extend(files)

    # Print the number of files detected
    print(f"{len(all_files)} file{'s' if len(all_files) != 1 else ''} detected")

    return all_files

In [ ]:
def parse_file_properties(file_path):
    """
    Extract animal ID, group, cortex, and condition from filenames like:
      H6b-sVNS-J7-2F_2025-02-10_18-12-31_sVNS_Contra
      H6b-VNS1-J7-3F_2025-04-10_18-12-31_Post-VNS_Peri
    """
    file_name = file_path[:-8]

    fname = os.path.basename(file_path)
    
    # 1. Extract animal ID (everything before the first date)
    animal_id = re.match(r"^(.*?_)\d{4}-\d{2}-\d{2}", fname)
    animal_id = animal_id.group(1)[:-1] if animal_id else fname.split("_")[0]
    
    # 2. Determine cortex
    cortex_match = re.search(r"(Contra|Peri)", fname, re.IGNORECASE)
    cortex = cortex_match.group(1).capitalize() if cortex_match else None
    
    # 3. Determine condition
    if "_Pre_" in fname:
        condition = "Pre"
    elif "_Post-sVNS_" in fname or "_Post-VNS_" in fname:
        condition = "Post"
    elif "_sVNS_" in fname or "_VNS_" in fname:
        condition = "VNS"
    else:
        condition = None
    
    # 4. Determine group (from animal_id)
    if "-sVNS" in animal_id:
        group = "sVNS"
    elif "-VNS" in animal_id:
        group = "VNS"
    else:
        group = None
    
    return animal_id, group, condition, cortex, file_name

## II. Retreiving data and performing calculations

In [ ]:
def burst_results_analysis(file_info):
    """
    Analyze precomputed burst data from one file and return summary results.

    Parameters
    ----------
    file_info : dict
        Dictionary containing file_name, ch_to_analyse, freq_to_analyse,
        min_cycle, epoch_idx, epoch_duration, etc.

    Returns
    -------
    burst_results : dict
        Summary statistics of burst properties for the analyzed file.
    """

    # ----------------------------------------------------------------------
    # Part 1: Load precomputed bursts and Curate bursts
    # ----------------------------------------------------------------------
    burst_path = f"{file_info['file_name']}_{file_info['ch_to_analyse']}_{file_info['freq_to_analyse']}_bursts_data.pkl"

    with open(burst_path, "rb") as f:
        bursts = pickle.load(f)

    # Convert bursts dictionary to DataFrame, keep only scalar 1D columns
    df_bursts = pd.DataFrame({
        k: np.array(v) for k, v in bursts.items() 
        if k in ['trial','peak_freq','peak_amp_iter','peak_amp_base','peak_time','fwhm_freq','fwhm_time']
    })

    # ----------------------------------------------------------------------
    # Part 2: Filter bursts by frequency of interest and min cycles
    # ----------------------------------------------------------------------
    freq_lims = file_info["Freq_Bands"][file_info["foi"]]
    fmin, fmax = freq_lims

    # Keep only bursts within the band
    df_bursts = df_bursts[(df_bursts["peak_freq"] >= fmin) & (df_bursts["peak_freq"] <= fmax)].reset_index(drop=True)

    # Keep only sufficiently long bursts
    if file_info["min_cycle"] is not None :
        df_bursts = df_bursts[
            df_bursts["fwhm_time"] > (file_info["min_cycle"] * (1 / df_bursts["peak_freq"]))
        ].reset_index(drop=True)

    # ----------------------------------------------------------------------
    # Part 3: Count bursts per epoch (with original epoch mapping)
    # ----------------------------------------------------------------------
    total_duration = len(file_info["epoch_idx"]) * file_info["epoch_duration"]  # seconds
    n_bursts_total = len(df_bursts)
    mean_n_burst = n_bursts_total / total_duration

    all_epochs = np.arange(file_info["initial_n_epoch"])
    kept_epochs = np.array(file_info["epoch_idx"])

    # Map CSV trial numbers (0..N-1) to original epochs
    trial_to_epoch = {trial: epoch for trial, epoch in enumerate(kept_epochs)}

    # Initialize burst counts dictionary
    epoch_burst_counts = {e: np.nan for e in all_epochs}

    # Count bursts per epoch
    for e in all_epochs:
        # Find which trial number corresponds to original epoch e
        trial_numbers = [trial for trial, epoch in trial_to_epoch.items() if epoch == e]

        if len(trial_numbers) == 0:
            epoch_burst_counts[e] = np.nan  # dropped epoch
        else:
            n_burst_e = np.sum(df_bursts["trial"] == trial_numbers[0])
            epoch_burst_counts[e] = n_burst_e

    # ----------------------------------------------------------------------
    # Part 4: Add new columns
    # ----------------------------------------------------------------------
    # Estimate Gaussian volume of each burst
    sigma_t = df_bursts["fwhm_time"] / (2 * np.sqrt(2 * np.log(2)))
    sigma_f = df_bursts["fwhm_freq"] / (2 * np.sqrt(2 * np.log(2)))
    df_bursts["gaus_vol_model"] = df_bursts["peak_amp_base"] * 2 * np.pi * sigma_t * sigma_f

    # Compute burst occurrence time across the full recording
    if file_info["condition"] == "VNS":
        df_bursts["peak_time_rec"] = (
            df_bursts["trial"] * (file_info["epoch_duration"]) +
            df_bursts["peak_time"] + 0.9 #artifact duration
        )
    else:
        df_bursts["peak_time_rec"] = (
            df_bursts["trial"] * (file_info["epoch_duration"]) +
            df_bursts["peak_time"]
        )
    
    # Map each burst's trial index (0..N_kept) to original epoch number
    trial_to_epoch = {trial: epoch for trial, epoch in enumerate(file_info["epoch_idx"])}
    # Apply correction to create a new column
    df_bursts["trial_idx"] = df_bursts["trial"].map(trial_to_epoch)

    # ----------------------------------------------------------------------
    # Part 4.5: Create single-burst-level table for mixed models
    # ----------------------------------------------------------------------

    # Define metadata columns (constant for all bursts in this file)
    meta_info = {
        "file_name": file_info["file_name"],
        "animal_id": file_info["animal_id"],
        "group": file_info["group"],
        "condition": file_info["condition"],
        "cortex": file_info["cortex"],
        "foi": file_info["foi"]
    }

    # Define which burst parameters to include
    burst_params = [
        "trial_idx","peak_freq", "peak_amp_iter", "peak_amp_base",
        "peak_time", "fwhm_freq", "fwhm_time", "gaus_vol_model", "peak_time_rec"
    ]

    # Create single-burst DataFrame by combining metadata and burst data
    single_burst_results = df_bursts[burst_params].copy()
    for key, value in meta_info.items():
        single_burst_results[key] = value

    # Optional: reorder columns for readability
    cols_order = list(meta_info.keys()) + burst_params
    single_burst_results = single_burst_results[cols_order]
    
    # ----------------------------------------------------------------------
    # Part 5: Compute per-epoch averages for each burst parameter (optimized)
    # ----------------------------------------------------------------------

    # Define which columns to summarize
    burst_metrics = [
        "peak_freq", "peak_amp_iter", "peak_amp_base",
        "peak_time", "fwhm_freq", "fwhm_time", "gaus_vol_model"
    ]

    # Prepare a dict to collect per-epoch means
    epoch_means = {metric: [] for metric in burst_metrics}

    for e in all_epochs:
        # Get all trials corresponding to this epoch
        trial_numbers = [trial for trial, epoch in trial_to_epoch.items() if epoch == e]

        if len(trial_numbers) == 0:
            # Dropped epoch → append NaN for all metrics
            for metric in burst_metrics:
                epoch_means[metric].append(np.nan)
        else:
            # Get bursts for the first trial in this epoch
            df_e = df_bursts[df_bursts["trial"] == trial_numbers[0]]

            if df_e.empty:
                # No bursts in this epoch
                for metric in burst_metrics:
                    epoch_means[metric].append(np.nan)
            else:
                # Compute mean per metric
                for metric in burst_metrics:
                    epoch_means[metric].append(df_e[metric].mean())

    # Unpack for later use
    epoch_mean_peak_freq = epoch_means["peak_freq"]
    epoch_mean_peak_amp_iter = epoch_means["peak_amp_iter"]
    epoch_mean_peak_amp_base = epoch_means["peak_amp_base"]
    epoch_mean_peak_time = epoch_means["peak_time"]
    epoch_mean_fwhm_freq = epoch_means["fwhm_freq"]
    epoch_mean_fwhm_time = epoch_means["fwhm_time"]
    epoch_mean_gaus_vol_model = epoch_means["gaus_vol_model"]

    # ----------------------------------------------------------------------
    # Part 6: Compute global averages across all bursts
    # ----------------------------------------------------------------------
    mean_peak_freq = df_bursts["peak_freq"].mean() if len(df_bursts) else np.nan
    mean_peak_amp_iter = df_bursts["peak_amp_iter"].mean() if len(df_bursts) else np.nan
    mean_peak_amp_base = df_bursts["peak_amp_base"].mean() if len(df_bursts) else np.nan
    mean_peak_time = df_bursts["peak_time"].mean() if len(df_bursts) else np.nan
    mean_fwhm_freq = df_bursts["fwhm_freq"].mean() if len(df_bursts) else np.nan
    mean_fwhm_time = df_bursts["fwhm_time"].mean() if len(df_bursts) else np.nan
    mean_gaus_vol_model = df_bursts["gaus_vol_model"].mean() if len(df_bursts) else np.nan

    # ----------------------------------------------------------------------
    # Part 7: Return results
    # ----------------------------------------------------------------------
    burst_results = {
        "file_name": file_info["file_name"],
        "animal_id": file_info["animal_id"],
        "group": file_info["group"],
        "condition": file_info["condition"],
        "cortex": file_info["cortex"],
        "foi": file_info["foi"],

        "n_bursts_total": n_bursts_total,
        "mean_n_burst": mean_n_burst,
        "epoch_burst_counts": epoch_burst_counts,

        "epoch_mean_peak_freq": epoch_mean_peak_freq,
        "epoch_mean_peak_amp_iter": epoch_mean_peak_amp_iter,
        "epoch_mean_peak_amp_base": epoch_mean_peak_amp_base,
        "epoch_mean_peak_time": epoch_mean_peak_time,
        "epoch_mean_fwhm_freq": epoch_mean_fwhm_freq,
        "epoch_mean_fwhm_time": epoch_mean_fwhm_time,
        "epoch_mean_gaus_vol_model" : epoch_mean_gaus_vol_model,

        "mean_peak_freq": mean_peak_freq,
        "mean_peak_amp_iter": mean_peak_amp_iter,
        "mean_peak_amp_base": mean_peak_amp_base,
        "mean_peak_time": mean_peak_time,
        "mean_fwhm_freq": mean_fwhm_freq,
        "mean_fwhm_time": mean_fwhm_time,
        "mean_gaus_vol_model": mean_gaus_vol_model,

        "gaus_vol_model" : df_bursts["gaus_vol_model"],
        "peak_time_rec" : df_bursts["peak_time_rec"]
    }

    return single_burst_results, burst_results 

In [ ]:
def fooof_results_analysis(file_info):
    """
    Analyze aperiodic (FOOOF) results exported to Excel and align with kept epochs.

    Parameters
    ----------
    file_info : dict
        Metadata containing:
            - file_name, ch_to_analyse, freq_to_analyse
            - initial_n_epoch (total number of epochs)
            - epoch_idx (list of kept epochs)
            - animal_id, group, condition, cortex

    Returns
    -------
    fooof_results : dict
        Contains per-epoch aperiodic parameters aligned with original epochs (NaN for dropped epochs).
    """

    # ----------------------------------------------------------------------
    # Part 1: Retrieve FOOOF data
    # ----------------------------------------------------------------------
    ch_to_analyse = file_info["ch_to_analyse"]
    freq_to_analyse = file_info["freq_to_analyse"]
    file_name = file_info["file_name"]

    excel_file = f"{file_name}_{ch_to_analyse}_{freq_to_analyse}_aperiodic_components.xlsx"

    # Read both sheets
    df_ap_fit = pd.read_excel(excel_file, sheet_name="Aperiodic Fit (ap)", index_col=0)
    df_ap_params = pd.read_excel(excel_file, sheet_name="Aperiodic Parameters", index_col=0)

    # ----------------------------------------------------------------------
    # Part 2: Align with kept and dropped epochs
    # ----------------------------------------------------------------------
    all_epochs = np.arange(file_info["initial_n_epoch"])   # e.g., 0..97
    kept_epochs = np.array(file_info["epoch_idx"])
    excel_columns = df_ap_params.columns.tolist()

    # Build aligned DataFrame efficiently (avoids fragmentation warnings)
    aligned_dict = {}

    # Map existing columns to their original epoch index
    for col_name, e in zip(excel_columns, kept_epochs):
        aligned_dict[f"Epoch_{e}"] = df_ap_params[col_name]

    # Fill missing (dropped) epochs with NaN columns
    missing_epochs = set(all_epochs) - set(kept_epochs)
    for e in missing_epochs:
        aligned_dict[f"Epoch_{e}"] = pd.Series([np.nan] * len(df_ap_params), index=df_ap_params.index)

    # Create final aligned DataFrame (avoids repeated .insert calls)
    aligned_params = pd.DataFrame(aligned_dict)[[f"Epoch_{e}" for e in all_epochs]]

    # Transpose → one row per epoch
    df_params_aligned = aligned_params.T
    df_params_aligned.index.name = "Epoch"

    # ----------------------------------------------------------------------
    # Part 3: Compute global averages (skip NaNs)
    # ----------------------------------------------------------------------
    mean_offset = df_params_aligned["Offset"].mean(skipna=True)
    mean_knee = df_params_aligned["Knee"].mean(skipna=True)
    mean_exponent = df_params_aligned["Exponent"].mean(skipna=True)

    # ----------------------------------------------------------------------
    # Part 4: Store structured results
    # ----------------------------------------------------------------------
    fooof_results = {
        "file_name": file_info["file_name"],
        "animal_id": file_info["animal_id"],
        "group": file_info["group"],
        "condition": file_info["condition"],
        "cortex": file_info["cortex"],
        "foi": file_info["foi"],

        # Per-epoch aligned values
        "epoch_offset": df_params_aligned["Offset"].tolist(),
        "epoch_knee": df_params_aligned["Knee"].tolist(),
        "epoch_exponent": df_params_aligned["Exponent"].tolist(),

        # Global averages
        "mean_offset": mean_offset,
        "mean_knee": mean_knee,
        "mean_exponent": mean_exponent,
    }

    return fooof_results


In [ ]:
def normalize_to_pre(df, value_cols, cortex_col="cortex", animal_col="animal_id",
                     condition_col="condition", info_cols=None):
    """
    Normalize one or more metrics per animal and cortex relative to 'Pre'.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input data with per-animal results.
    value_cols : list of str
        Columns to normalize (can be scalar or per-epoch arrays).
    cortex_col : str
        Column identifying cortex (default 'cortex').
    animal_col : str
        Column identifying animal (default 'animal_id').
    condition_col : str
        Column identifying condition (default 'condition').
    info_cols : list of str, optional
        Metadata columns to keep. Default:
        ["file_name","animal_id","group","condition","cortex"]

    Returns
    -------
    df_norm : pd.DataFrame
        Copy of df with only info_cols and <value_col>_norm for each metric.
    """
    
    if info_cols is None:
        info_cols = ["file_name","animal_id","group","condition","cortex","foi"]
    
    df_norm_list = []

    for (animal, cortex), subset in df.groupby([animal_col, cortex_col]):
        pre_rows = subset[subset[condition_col] == "Pre"]
        if pre_rows.empty:
            warnings.warn(f"No Pre condition found for {animal}, {cortex} - skipping normalization.")
            for _, row in subset.iterrows():
                norm_row = {col: row[col] for col in info_cols}
                for col in value_cols:
                    norm_row[f"{col}_norm"] = np.nan if not isinstance(row[col], (list, np.ndarray)) else np.full_like(np.array(row[col], dtype=float), np.nan)
                df_norm_list.append(norm_row)
            continue

        # Compute mean of Pre for each value_col
        mean_pre_dict = {}
        norm_pre_dict = {}
        for col in value_cols:
            pre_val = pre_rows[col].iloc[0]
            if isinstance(pre_val, (list, np.ndarray)):
                pre_val = np.array(pre_val, dtype=float)
                mean_pre = np.nanmean(pre_val)
                if mean_pre == 0 or np.isnan(mean_pre):
                    warnings.warn(f"Mean Pre = 0 or NaN for {animal}, {cortex}, column {col} - setting NaN.")
                    norm_pre = np.full_like(pre_val, np.nan)
                else:
                    norm_pre = pre_val * 100 / mean_pre - 100
            else:
                mean_pre = float(pre_val)
                norm_pre = 0 if mean_pre != 0 else np.nan
            mean_pre_dict[col] = mean_pre
            norm_pre_dict[col] = norm_pre

        # Normalize all rows
        for _, row in subset.iterrows():
            norm_row = {col: row[col] for col in info_cols}
            for col in value_cols:
                val = row[col]
                mean_pre = mean_pre_dict[col]
                if isinstance(val, (list, np.ndarray)):
                    val = np.array(val, dtype=float)
                    if row[condition_col] == "Pre":
                        norm_val = norm_pre_dict[col]
                    else:
                        if mean_pre == 0 or np.isnan(mean_pre):
                            warnings.warn(f"Division by zero for {animal}, {cortex}, column {col} - setting NaN.")
                            norm_val = np.full_like(val, np.nan)
                        else:
                            norm_val = (val * 100 / mean_pre) - 100
                else:
                    if row[condition_col] == "Pre":
                        norm_val = norm_pre_dict[col]
                    else:
                        if mean_pre == 0 or np.isnan(mean_pre):
                            warnings.warn(f"Division by zero for {animal}, {cortex}, column {col} - setting NaN.")
                            norm_val = np.nan
                        else:
                            norm_val = (val * 100 / mean_pre) - 100
                norm_row[f"{col}_norm"] = norm_val
            df_norm_list.append(norm_row)

    df_norm = pd.DataFrame(df_norm_list)
    return df_norm


In [ ]:
def zscore_to_pre(df, value_cols, group_col="group", cortex_col="cortex",
                            condition_col="condition", info_cols=None):
    """
    Z-score one or more metrics per cortex and group relative to the 'Pre' condition (across animals).

    For each (group, cortex):
    - Compute mean_pre and std_pre from all animals' Pre condition values.
    - Apply (val - mean_pre) / std_pre to Pre, VNS, and Post for all animals in that group & cortex.
    - Works for scalar values or arrays (per-epoch metrics).
    - NaNs are preserved, and division-by-zero (std_pre=0) triggers a warning + NaN.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing per-animal metrics.
    value_cols : list of str
        List of columns to z-score (each can be a scalar or array).
    group_col : str, default="group"
        Column defining experimental group (e.g., "VNS", "sVNS").
    cortex_col : str, default="cortex"
        Column defining cortex type (e.g., "Contra", "Peri").
    condition_col : str, default="condition"
        Column defining condition (e.g., "Pre", "VNS", "Post").
    info_cols : list of str, optional
        Metadata columns to preserve. Default: ["file_name","animal_id","group","condition","cortex"]

    Returns
    -------
    df_zscore : pd.DataFrame
        Copy of df with only info_cols and new columns <value_col>_zscore for each metric.
    """

    if info_cols is None:
        info_cols = ["file_name", "animal_id", "group", "condition", "cortex","foi"]

    df_zscore_list = []

    # Process group × cortex combinations
    for (group, cortex), subset in df.groupby([group_col, cortex_col]):
        pre_rows = subset[subset[condition_col] == "Pre"]
        if pre_rows.empty:
            warnings.warn(f"No Pre condition found for group={group}, cortex={cortex} — skipping z-scoring.")
            for _, row in subset.iterrows():
                z_row = {col: row[col] for col in info_cols}
                for col in value_cols:
                    val = row[col]
                    z_row[f"{col}_zscore"] = (
                        np.full_like(np.array(val, dtype=float), np.nan)
                        if isinstance(val, (list, np.ndarray))
                        else np.nan
                    )
                df_zscore_list.append(z_row)
            continue

        # Compute group-level Pre mean & std
        mean_pre_dict = {}
        std_pre_dict = {}

        for col in value_cols:
            pre_vals = pre_rows[col].values

            # Stack arrays or just collect scalars
            if isinstance(pre_vals[0], (list, np.ndarray)):
                pre_arrays = np.stack(pre_vals)
                mean_pre = np.nanmean(pre_arrays, axis=0)
                std_pre = np.nanstd(pre_arrays, axis=0)
            else:
                pre_arrays = np.array(pre_vals, dtype=float)
                mean_pre = np.nanmean(pre_arrays)
                std_pre = np.nanstd(pre_arrays)

            mean_pre_dict[col] = mean_pre
            std_pre_dict[col] = std_pre

        # Apply z-scoring to all conditions for this group × cortex
        for _, row in subset.iterrows():
            z_row = {col: row[col] for col in info_cols}
            for col in value_cols:
                val = row[col]
                mean_pre = mean_pre_dict[col]
                std_pre = std_pre_dict[col]

                if isinstance(val, (list, np.ndarray)):
                    val = np.array(val, dtype=float)
                    if np.any(std_pre == 0) or np.all(np.isnan(std_pre)):
                        warnings.warn(f"std(Pre)=0 or NaN for {col}, group={group}, cortex={cortex} — setting NaN.")
                        z_val = np.full_like(val, np.nan)
                    else:
                        z_val = (val - mean_pre) / std_pre
                else:
                    if std_pre == 0 or np.isnan(std_pre):
                        z_val = np.nan
                    else:
                        z_val = (val - mean_pre) / std_pre

                z_row[f"{col}_zscore"] = z_val

            df_zscore_list.append(z_row)

    df_zscore = pd.DataFrame(df_zscore_list)
    return df_zscore

## III. Plotting results and export

In [ ]:
def plot_violin_all_burst(df, value_col, group_col="cortex", hue_col="condition",
                           group_order=None, hue_order=None, group_to_test=None, title=None,
                           custom_palette=None, export_path=None, pairs=None,
                           family=Poisson(), text_format='star', loc='inside'):
    """
    Plot burst-level data with split violin plots and GEE statistical tests.

    Each burst is included, and correlation between bursts within animals is handled via a
    Generalized Estimating Equation (GEE) with an exchangeable correlation structure.

    Since our data contains multiple bursts per subject (repeated measures), standard ANOVA 
    assumptions of independence are violated. We use Generalized Estimating Equations (GEE) 
    with an exchangeable correlation structure to robustly model the effect of the condition 
    while accounting for intra-subject correlations.

    Parameters
    ----------
    df : pd.DataFrame
        Burst-level dataframe containing at least `value_col`, `hue_col`, `group_col`, and `animal_id`.
    value_col : str
        Name of the column to plot on the y-axis and analyze statistically.
    group_col : str, default="cortex"
        Categorical variable defining the x-axis grouping.
    hue_col : str, default="condition"
        Variable defining the violin split and used for GEE modeling.
    group_order : list of str, optional
        Order of categories on the x-axis.
    hue_order : list of str, optional
        Order of hue categories.
    group_to_test : str, optional
        Filter dataframe to this group before plotting.
    title : str, optional
        Custom plot title. If None, a default title is generated.
    custom_palette : dict, optional
        Custom color palette for hue categories.
    export_path : str, optional
        Directory to save the figure.
    pairs : list of tuples, optional
        List of hue_col value pairs to perform GEE comparisons on.
    family : statsmodels family, default=Poisson()
        Family to use for the GEE (can be Gaussian, Poisson, Gamma, etc.).
    text_format : str, default='star'
        Format for statistical annotation text ('star' or 'full').
    loc : str, default='inside'
        Location of statistical annotation text.

    Returns
    -------
    None
        Displays the violin plot and optionally exports it as an image file.
    """

    # --- Filter by experimental group if requested ---
    if group_to_test is not None:
        df = df[df["group"] == group_to_test]

    # --- Custom color palette ---
    if custom_palette is None:
        custom_palette = {"Pre": "#2ca02c", "VNS": "#9467bd", "Post": "#ff7f0e"}

    # --- Create the plot ---
    fig, ax = plt.subplots(figsize=(7, 5))

    sns.violinplot(
        data=df, x=group_col, y=value_col, hue=hue_col,
        split=True, inner="quart", fill=False,
        palette=custom_palette,
        order=group_order, hue_order=hue_order, ax=ax
    )

    sns.stripplot(
        data=df, x=group_col, y=value_col, hue=hue_col,
        dodge=True, palette=["black"] * len(df[hue_col].unique()),
        size=2.5, jitter=True, alpha=0.5,
        order=group_order, hue_order=hue_order, ax=ax, legend=False
    )

    # --- Run GEE for each pair ---
    if pairs:
        y_max = df[value_col].max()
        y_min = df[value_col].min()
        y_offset = (y_max - y_min) * 0.1

        for i, (val1, val2) in enumerate(pairs):
            df_pair = df[df[hue_col].isin([val1, val2])]
            if len(df_pair) > 0:
                formula = f"{value_col} ~ {hue_col}"
                try:
                    model = GEE.from_formula(
                        formula=formula,
                        groups="animal_id",
                        cov_struct=Exchangeable(),
                        family=family,
                        data=df_pair
                    )
                    result = model.fit()
                    p = result.pvalues.get(hue_col + f"[T.{val2}]", np.nan)
                    star = (
                        "***" if p < 0.001 else
                        "**" if p < 0.01 else
                        "*" if p < 0.05 else "ns"
                    )
                    ax.text(i, y_max + y_offset, star, ha="center", fontsize=14, weight="bold")
                except Exception as e:
                    print(f"⚠️ GEE failed for pair {val1}-{val2}: {e}")

    # --- Title and labels ---
    if title is None:
        title = f"{value_col} (burst-level GEE - Violin)"
    ax.set_title(title, fontsize=13, weight='bold')
    ax.set_ylabel(value_col, fontsize=11)
    ax.set_xlabel(group_col, fontsize=11)

    # --- Legend ---
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:len(hue_order)], labels[:len(hue_order)],
                  title=hue_col, loc='center left', bbox_to_anchor=(1, 0.5))

    plt.tight_layout()
    plt.show()

    # --- Export section ---
    if export_path is not None:
        os.makedirs(export_path, exist_ok=True)
        group_part = f"_{group_to_test}" if group_to_test else ""
        foi_part = f"_{df['foi'].iloc[0]}" if 'foi' in df.columns else ""
        filename = f"{value_col}_GEE_Violin{group_part}{foi_part}.png"
        save_path = os.path.join(export_path, filename)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure exported to: {save_path}")

In [ ]:
def plot_bar_all_burst(df, value_col, group_col="cortex", hue_col="condition",
                       group_order=None, hue_order=None, group_to_test=None, title=None,
                       custom_palette=None, export_path=None, pairs=None,
                       family=Poisson(), text_format='star', loc='inside'):
    """
    Plot burst-level data with GEE statistical tests, optionally using pairwise comparisons.

    Each burst is included, and correlation between bursts within animals is handled via a 
    Generalized Estimating Equation (GEE) with an exchangeable correlation structure.

    Parameters
    ----------
    df : pd.DataFrame
        Burst-level dataframe containing at least `value_col`, `hue_col`, `group_col`, and `animal_id`.
    value_col : str
        Name of the column to plot on the y-axis and analyze statistically.
    group_col : str, default="cortex"
        Categorical variable defining the x-axis grouping.
    hue_col : str, default="condition"
        Variable defining the bar colors and used for GEE modeling.
    group_order : list of str, optional
        Order of categories on the x-axis.
    hue_order : list of str, optional
        Order of hue categories.
    group_to_test : str, optional
        Filter dataframe to this group before plotting.
    title : str, optional
        Custom plot title. If None, a default title is generated.
    custom_palette : dict, optional
        Custom color palette for hue categories.
    export_path : str, optional
        Directory to save the figure.
    pairs : list of tuples, optional
        List of hue_col value pairs to perform GEE comparisons on (e.g., [('Pre','VNS')]).
    family : statsmodels family, default=Poisson()
        Family to use for the GEE (can be Gaussian, Poisson, Gamma, etc.).
    text_format : str, default='star'
        Format for statistical annotation text ('star' or 'full').
    loc : str, default='inside'
        Location of statistical annotation text.

    Returns
    -------
    None
        Displays the barplot and optionally exports it as an image file.
    """

    # --- Filter by experimental group if requested ---
    if group_to_test is not None:
        df = df[df["group"] == group_to_test]

    # --- Custom color palette ---
    if custom_palette is None:
        custom_palette = {"Pre": "#2ca02c", "VNS": "#9467bd", "Post": "#ff7f0e"}

    # --- Create the plot ---
    fig, ax = plt.subplots(figsize=(7, 5))

    sns.barplot(
        data=df, x=group_col, y=value_col, hue=hue_col,
        errorbar="se", alpha=0.8,
        order=group_order, hue_order=hue_order,
        palette=custom_palette, ax=ax
    )
    sns.stripplot(
        data=df, x=group_col, y=value_col, hue=hue_col,
        dodge=True, palette=["black"] * len(df[hue_col].unique()),
        size=2.5, jitter=True, alpha=0.5,
        order=group_order, hue_order=hue_order, ax=ax, legend=False
    )

    # --- Run GEE for each pair ---
    if pairs:
        y_max = df[value_col].max()
        y_min = df[value_col].min()
        y_offset = (y_max - y_min) * 0.1

        for i, (val1, val2) in enumerate(pairs):
            df_pair = df[df[hue_col].isin([val1, val2])]
            if len(df_pair) > 0:
                formula = f"{value_col} ~ {hue_col}"
                try:
                    model = GEE.from_formula(
                        formula=formula,
                        groups="animal_id",
                        cov_struct=Exchangeable(),
                        family=family,
                        data=df_pair
                    )
                    result = model.fit()
                    p = result.pvalues.get(hue_col + f"[T.{val2}]", np.nan)
                    star = (
                        "***" if p < 0.001 else
                        "**" if p < 0.01 else
                        "*" if p < 0.05 else "ns"
                    )
                    ax.text(i, y_max + y_offset, star, ha="center", fontsize=14, weight="bold")
                except Exception as e:
                    print(f"⚠️ GEE failed for pair {val1}-{val2}: {e}")

    # --- Title and labels ---
    if title is None:
        title = f"{value_col} (burst-level GEE)"
    ax.set_title(title, fontsize=13, weight='bold')
    ax.set_ylabel(value_col, fontsize=11)
    ax.set_xlabel(group_col, fontsize=11)

    # --- Legend ---
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:len(hue_order)], labels[:len(hue_order)],
                  title=hue_col, loc='center left', bbox_to_anchor=(1, 0.5))

    plt.tight_layout()
    plt.show()

    # --- Export section ---
    if export_path is not None:
        os.makedirs(export_path, exist_ok=True)
        group_part = f"_{group_to_test}" if group_to_test else ""
        foi_part = f"_{df['foi'].iloc[0]}" if 'foi' in df.columns else ""
        filename = f"{value_col}_GEE{group_part}{foi_part}.png"
        save_path = os.path.join(export_path, filename)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure exported to: {save_path}")

In [ ]:
def plot_bar(df, value_col, group_col="cortex", hue_col="condition",
             group_order=None, hue_order=None, group_to_test=None, title=None, custom_palette=None,
             export_path=None, pairs=None, test='t-test_ind', text_format='star', loc='inside'):
    """
    Plot grouped barplots with individual data points and optional statistical annotations.

    Each bar represents the mean ± SE of a metric (e.g., mean_peak_freq) per animal, grouped
    by a categorical variable such as cortex or condition. Optionally, the function can:
    - Filter data by a specific stimulation group (e.g., VNS or sVNS)
    - Add per-animal data points (stripplot)
    - Add statistical test annotations between selected pairs
    - Export the figure automatically to a file path

    Aesthetic customization:
    - Fixed legend always placed to the right of the plot (never overlapping)
    - Custom condition colors: Pre=green, VNS=purple, Post=orange

    Parameters
    ----------
    df : pandas.DataFrame
        Input data table containing the metric and metadata.

    value_col : str
        Name of the column to plot on the y-axis.

    group_col : str, default="cortex"
        Categorical variable defining the x-axis grouping.

    hue_col : str, default="condition"
        Variable defining the bar colors (e.g. "Pre", "VNS", "Post").

    group_order, hue_order : list, optional
        Order of appearance for the x-axis groups and hue levels.

    group_to_test : str, optional
        If specified, filters data to only include this experimental group (e.g. "VNS" or "sVNS").

    title : str, optional
        Custom plot title. If None, an automatic descriptive title is generated.
    
    custom_palette : dict, optional
        Custom color palette.

    export_path : str, optional
        Directory to save the figure as a PNG file.

    pairs : list of tuples, optional
        List of pairwise comparisons to annotate statistically.

    test : str, default='t-test_ind'
        Statistical test used for pairwise comparisons.

    text_format : str, default='star'
        Format of statistical annotation text ('star' or 'full').

    loc : str, default='inside'
        Location of statistical annotation text.

    Returns
    -------
    None
        Displays the barplot and optionally exports it as an image file.
    """

    # --- Filter by experimental group if requested ---
    if group_to_test is not None:
        df = df[df["group"] == group_to_test]

    # --- Auto title generation ---
    if title is None:
        parts = [value_col, f"by {group_col}"]
        if hue_col:
            parts.append(f"and {hue_col}")
            if group_to_test is not None:
                parts.append(f"for {group_to_test} group")
        title = " ".join(parts)

    # --- Custom color palette for the 3 conditions ---
    if custom_palette is None : 
        custom_palette = {
            "Pre": "#2ca02c",     # green
            "VNS": "#9467bd",     # purple
            "Post": "#ff7f0e"     # orange
        }

    fig, ax = plt.subplots(figsize=(7, 5))

    sns.barplot(
        data=df, x=group_col, y=value_col, hue=hue_col,
        errorbar="se", alpha=0.8,
        order=group_order, hue_order=hue_order,
        palette=custom_palette, ax=ax
    )

    sns.stripplot(
        data=df, x=group_col, y=value_col, hue=hue_col,
        dodge=True, palette=["black"]*len(df[hue_col].unique()),
        size=5, jitter=True,
        order=group_order, hue_order=hue_order, ax=ax, legend=False
    )

    # --- Title and labels ---
    ax.set_title(title, fontsize=13, weight='bold')
    ax.set_ylabel(value_col, fontsize=11)
    ax.set_xlabel(group_col, fontsize=11)

    # --- Legend: always on the right side ---
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:len(hue_order)], labels[:len(hue_order)],
                  title=hue_col, loc='center left', bbox_to_anchor=(1, 0.5))

    # --- Statistical annotations (if any) ---
    if pairs is not None:
        annotator = Annotator(
            ax, pairs, data=df, x=group_col, y=value_col, hue=hue_col,
            order=group_order, hue_order=hue_order
        )
        annotator.configure(test=test, text_format=text_format, loc=loc)
        annotator.apply_and_annotate()

    plt.tight_layout()
    plt.show()

    # --- Export section ---
    if export_path is not None:
        import os
        os.makedirs(export_path, exist_ok=True)
        cortex_part = f"_{'_'.join(group_order)}" if group_order else ""
        group_part = f"_{group_to_test}" if group_to_test else ""
        foi_part = f"_{df['foi'].iloc[0]}" if 'foi' in df.columns else ""
        filename = f"{value_col}{cortex_part}{group_part}{foi_part}.png"
        save_path = os.path.join(export_path, filename)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure exported to: {save_path}")


In [ ]:
def plot_epoch_line(df, array_col, hue_col="condition",
                    hue_order=None, filter_col=None, filter_val=None,
                    group_col=None, group_order=None, title=None, export_path=None):
    """
    Plot mean ± SEM (standard error of the mean) curves for an epoch-wise metric,
    where each curve represents the average of a subgroup (e.g., condition or group)
    across all animals.

    Conceptually:
    - Each *animal* provides one array of values per epoch (stored in `array_col`).
    - The function groups these arrays by `hue_col` (e.g. condition = Pre, VNS, Post).
    - Optionally, a second grouping (`group_col`) can split curves further
      (e.g. VNS vs sVNS animals).
    - Curves are plotted sequentially on the x-axis by `hue_order`, instead of overlapping.

    Example use cases:
    - Compare Pre, VNS, and Post sequentially for one cortex.
    - Compare VNS vs sVNS animals within a single cortex.
    - Compare Contra vs Peri cortices within a single experimental group.

    Parameters
    ----------
    df : pandas.DataFrame
        Table where each row corresponds to one animal (or file),
        containing metadata columns and at least one column (`array_col`)
        that stores a per-epoch array or list of metric values.

    array_col : str
        Name of the column containing per-epoch arrays (e.g. "epoch_mean_peak_amp_base").
        Each element is an array of equal length representing a value per epoch for that animal.

    hue_col : str, default="condition"
        Column used to define which variable determines each *set of curves* (color-coded).
        Example: "condition" → Pre, VNS, Post lines.

    hue_order : list of str, optional
        Custom order for the `hue_col` levels (e.g., ["Pre", "VNS", "Post"]).
        Determines the order in which conditions appear *sequentially* on the x-axis.

    filter_col : str, optional
        Column name to filter the dataset before plotting (e.g., "cortex" or "group").
        Only rows where `df[filter_col] == filter_val` are kept.

    filter_val : str, optional
        Value used to subset the DataFrame (e.g., filter_col="cortex", filter_val="Contra").

    group_col : str, optional
        Secondary grouping factor to compare within each hue (e.g., "group" for VNS vs sVNS).
        Each value of `group_col` will produce its own curve *within* each hue block.

    group_order : list of str, optional
        Custom display order for `group_col` values (e.g., ["VNS", "sVNS"]).

    title : str, optional
        Figure title displayed at the top.

    Notes
    -----
    - The x-axis represents concatenated epochs (e.g. Pre epochs first, then VNS, then Post).
    - The y-axis represents the mean ± SEM of `array_col` values across animals.
    - Each line corresponds to one subgroup (e.g., VNS-Pre, sVNS-Pre, etc.).
    - Shaded areas show ±1 SEM (standard error of the mean) across animals.

    Returns
    -------
    None
        Displays a matplotlib plot directly.
    """
# --- Auto title generation ---
    if title is None:
        parts = [array_col, f"by {group_col}"]
        if hue_col:
            parts.append(f"and {hue_col}")
        parts.append(f"in {filter_val}")
        title = " ".join(parts)

    # Apply filtering
    if filter_col and filter_val:
        df = df[df[filter_col] == filter_val]

    fig, ax = plt.subplots(figsize=(10,5))

    if hue_order is None:
        hue_order = df[hue_col].unique()
    if group_col and group_order is None:
        group_order = df[group_col].unique()

    # Define custom colors
    hue_colors = {
        "VNS": "#9467bd",       # purple
        "sVNS": "#2f2f2f",      # dark grey
        # other conditions will use default palette
    }
    palette = sns.color_palette("Set2", len(group_order) if group_col else 1)

    # Track where each condition starts
    total_epochs = 0
    vns_start, vns_end = None, None

    for cond in hue_order:
        subset_cond = df[df[hue_col] == cond]
        if len(subset_cond) == 0:
            continue

        if group_col:
            for i, grp in enumerate(group_order):
                subset_grp = subset_cond[subset_cond[group_col] == grp]
                if len(subset_grp) == 0:
                    continue

                data_array = np.stack(subset_grp[array_col].values)
                mean_curve = np.nanmean(data_array, axis=0)
                sem_curve = np.nanstd(data_array, axis=0) / np.sqrt(data_array.shape[0])

                x = np.arange(len(mean_curve)) + total_epochs

                # Custom color
                if grp == "VNS":
                    color = "#9467bd"
                elif grp == "sVNS":
                    color = "#2f2f2f"
                else:
                    color = palette[i]

                ax.plot(x, mean_curve, label=f"{cond} - {grp}", color=color)
                ax.fill_between(x, mean_curve-sem_curve, mean_curve+sem_curve,
                                alpha=0.3, color=color)

        else:
            data_array = np.stack(subset_cond[array_col].values)
            mean_curve = np.nanmean(data_array, axis=0)
            sem_curve = np.nanstd(data_array, axis=0) / np.sqrt(data_array.shape[0])

            x = np.arange(len(mean_curve)) + total_epochs
            ax.plot(x, mean_curve, label=cond)
            ax.fill_between(x, mean_curve-sem_curve, mean_curve+sem_curve, alpha=0.3)

        # Save VNS start/end for shading
        if cond == "VNS":
            vns_start = total_epochs
            vns_end = total_epochs + len(mean_curve)

        total_epochs += len(mean_curve)

    # Add VNS shaded rectangle after loop
    if vns_start is not None and vns_end is not None:
        ax.axvspan(vns_start, vns_end, color='lightgrey', alpha=0.2)
        ax.text((vns_start + vns_end)/2, ax.get_ylim()[1]*0.95, "VNS/sVNS",
                ha='center', va='top', fontsize=10, color='black')

    ax.set_xlabel("Epochs (concatenated by condition)")
    ax.set_ylabel(array_col)
    if title:
        ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

    if export_path is not None:
        os.makedirs(export_path, exist_ok=True)
        cortex_part = f"_{filter_val}" if filter_val else ""
        foi_part = f"_{df['foi'].iloc[0]}" if 'foi' in df.columns else ""
        filename = f"{array_col}_{cortex_part}_{foi_part}.png"
        save_path = os.path.join(export_path, filename)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure exported to: {save_path}")

In [ ]:
def plot_several_columns(df_data, columns_single, columns_array,export_path = None):
    """
    Generate plots for multiple metrics across conditions and cortical regions.

    Plot are generated for each column for:
    - Raw data 
    - Normalized values (computed via `normalize_to_pre`) are plotted to show % change.
    - Z-scored values (computed via `zscore_to_pre`) are plotted to show standard deviation change.
    - Statistical comparisons are added between relevant condition pairs.

    Parameters
    ----------
    df_data : pd.DataFrame
        Input dataframe containing the data to plot (e.g., df_burst or df_fooof).
    columns_single : list of str
        Columns containing scalar metrics (one value per condition).
    columns_array : list of str
        Columns containing array metrics (e.g., per-epoch data).
    export_path : str
        Path where the plots are gonna be exported.
    
    Notes
    -----
    - The function automatically normalizes and Z-scores all specified columns.
    - Plots are organized by cortex ('Contra', 'Peri'), condition ('Pre', 'VNS', 'Post'),
      and group ('sVNS', 'VNS').
    - Statistical tests (default: independent t-test) are automatically applied to 
      predefined pairs of conditions.
    """

    # -----------------------------
    # Define plotting parameters
    # -----------------------------

    # Order of cortices to display on the x-axis
    cortex_order = ["Contra", "Peri"]

    # Order of experimental conditions
    condition_order = ["Pre", "VNS", "Post"]

    # Order of stimulation groups (used to loop over separate plots)
    group_order = ["VNS", "sVNS"]

    # Define pairs of (cortex, condition) combinations to compare statistically in the plots
    pairs = [
        (("Contra", "Pre"), ("Contra", "VNS")),
        (("Contra", "Pre"), ("Contra", "Post")),
        (("Peri", "Pre"), ("Peri", "VNS")),
        (("Peri", "Pre"), ("Peri", "Post")),
        (("Contra", "Pre"), ("Peri", "Pre"))
    ]

    pairs_VNS_vs_sVNS = [
        (("Contra", "VNS"), ("Contra", "sVNS")),
        (("Peri", "VNS"), ("Peri", "sVNS")),
    ]

    # Combine single-value and array-based columns into one array of all columns to normalize
    all_columns = np.concatenate((columns_single, columns_array))

    # -----------------------------
    # Compute normalization and z-scoring
    # -----------------------------

    # Normalize all specified columns relative to the 'Pre' condition
    # (adds _norm columns for each metric)
    df_data_norm = normalize_to_pre(
        df_data,
        value_cols=all_columns
    )

    # Z-score all specified columns relative to the 'Pre' condition
    # (adds _zscore columns for each metric)
    df_data_zscore = zscore_to_pre(
        df_data,
        value_cols=columns_single
    )

    # -----------------------------
    # Generate bar plots for single-value columns
    # -----------------------------

    # Loop through each scalar metric column (not epoch arrays)
    for column in columns_single:
        
        # Comparison of VNS vs sVNS for normalized post-VNS value data plot (evolution in %)
        df_data_norm_post = df_data_norm[df_data_norm["condition"] == "Post"]

        custom_palette_condition =  {
        "VNS": "#9467bd",       # purple
        "sVNS": "#2f2f2f"      # dark grey
        }
        plot_bar(
            df_data_norm_post,
            value_col=f"{column}_norm",
            group_col="cortex",
            hue_col="group",
            group_order=cortex_order,
            hue_order=group_order,
            pairs=pairs_VNS_vs_sVNS,
            test='t-test_ind',
            group_to_test=None,
            custom_palette = custom_palette_condition,
            export_path = export_path
        )

        # Comparison of peri vs contra pre-VNS

        df_data_pre = df_data[df_data["condition"] == "Pre"]
        plot_bar(
                df_data_pre,
                value_col=column,
                group_col="cortex",
                hue_col="condition",
                group_order=cortex_order,
                hue_order=["Pre"],
                pairs=[(("Contra", "Pre"), ("Peri", "Pre"))],
                test='t-test_ind',
                group_to_test=None,
                export_path = export_path
        )
        
        # Create plots for each stimulation group (sVNS and VNS)
        for group in group_order:
            # Raw data plot
            # Plot bar chart of the raw values per cortex and condition
            plot_bar(
                df_data,
                value_col=column,
                group_col="cortex",
                hue_col="condition",
                group_order=cortex_order,
                hue_order=condition_order,
                pairs=pairs,
                test='t-test_ind',
                group_to_test=group,
                export_path = export_path
            )

            # Normalized data plot
            # Plot the same metric normalized to its Pre value
            plot_bar(
                df_data_norm,
                value_col=f"{column}_norm",
                group_col="cortex",
                hue_col="condition",
                group_order=cortex_order,
                hue_order=condition_order,
                pairs=pairs,
                test='t-test_ind',
                group_to_test=group,
                export_path = export_path
            )

            # Z-scored data plot
            # Plot the same metric Z-scored relative to Pre
            plot_bar(
                df_data_zscore,
                value_col=f"{column}_zscore",
                group_col="cortex",
                hue_col="condition",
                group_order=cortex_order,
                hue_order=condition_order,
                pairs=pairs,
                test='t-test_ind',
                group_to_test=group,
                export_path = export_path
            )
            
            
    # -----------------------------
    # Generate line plots for array columns
    # -----------------------------

    # Loop through each scalar metric column (not epoch arrays)
    for column in columns_array:
        # Create plots for each stimulation group (sVNS and VNS)
        for group in group_order:
            # Raw data plot
            # Plot line chart of the raw values per cortex and condition
            for cortex in cortex_order:
                plot_epoch_line(
                    df_data,
                    array_col=column,
                    hue_col="condition",               # → Pre, VNS, Post define the 3 sequential blocks
                    hue_order=condition_order,
                    group_col="group",                 # → two groups: VNS and sVNS → two curves
                    group_order=group_order,
                    filter_col="cortex",               # → select one cortex
                    filter_val= cortex,
                    export_path = export_path
                )

                # Normalized data plot
                # Plot the same metric normalized to its Pre value
                plot_epoch_line(
                    df_data_norm,
                    array_col=f"{column}_norm",
                    hue_col="condition",               # → Pre, VNS, Post define the 3 sequential blocks
                    hue_order=condition_order,
                    group_col="group",                 # → two groups: VNS and sVNS → two curves
                    group_order=group_order,
                    filter_col="cortex",               # → select one cortex
                    filter_val=cortex,
                    export_path = export_path
                )

In [ ]:
def plot_several_columns_all_burst(df_data, columns_to_analyze, export_path=None):
    """
    Run burst-level mixed model (LMM) analysis and plots across several metrics.

    Parameters
    ----------
    df_data : pd.DataFrame
        DataFrame containing all burst-level data (e.g., df_single_bursts)
    columns_to_analyze : list of str
        List of burst metrics to analyze (e.g. ["peak_freq", "peak_amp_iter", ...])
    export_path : str, optional
        Path where plots will be exported.
    """

    # -----------------------------
    # Define plotting parameters
    # -----------------------------

    cortex_order = ["Contra", "Peri"]
    condition_order = ["Pre", "VNS", "Post"]
    group_order = ["VNS", "sVNS"]

    # Custom colors for the 3 conditions
    custom_palette = {
        "Pre": "#2ca02c",     # green
        "VNS": "#9467bd",     # purple
        "Post": "#ff7f0e"     # orange
    }

    custom_palette_condition =  {
        "VNS": "#9467bd",       # purple
        "sVNS": "#2f2f2f"      # dark grey
    }

    # Pair of data to statistically compare
    pairs = [
        (("Contra", "Pre"), ("Contra", "VNS")),
        (("Contra", "Pre"), ("Contra", "Post")),
        (("Peri", "Pre"), ("Peri", "VNS")),
        (("Peri", "Pre"), ("Peri", "Post")),
        (("Contra", "Pre"), ("Peri", "Pre"))
    ]

    pairs_VNS_vs_sVNS = [
        (("Contra", "VNS"), ("Contra", "sVNS")),
        (("Peri", "VNS"), ("Peri", "sVNS")),
    ]

    # -----------------------------
    # Run GEE and plot per column
    # -----------------------------            
    for column in columns_to_analyze:
        print(f"\n{'='*80}\nAnalyzing {column} with mixed models...\n{'='*80}")

        #Comparison sVNS vs VNS post
        df_data_post = df_data[df_data["condition"] == "Post"]
        plot_violin_all_burst(
            df_data_post,
            value_col=column,
            group_col="cortex",
            hue_col="group",
            group_order=cortex_order,
            hue_order=group_order,
            group_to_test=None,
            custom_palette = custom_palette_condition,
            export_path = export_path,
            pairs=pairs_VNS_vs_sVNS
        )

                # --- Violin plot ---
        plot_violin_all_burst(
            df=df_group,
            value_col=column,
            group_col="cortex",
            hue_col="condition",
            group_order=cortex_order,
            hue_order=condition_order,
            group_to_test=group,
            title=f"{column} - Violin ({group})",
            custom_palette=None,
            export_path=export_path,
            pairs=pairs
        )

        # --- For each stimulation group (VNS and sVNS separately) ---
        for group in group_order:
            df_group = df_data[df_data["group"] == group].copy()
            if df_group.empty:
                print(f"⚠️ No data found for group '{group}', skipping...")
                continue

            # --- Bar plot ---
            plot_bar_all_burst(
                df=df_group,
                value_col=column,
                group_col="cortex",
                hue_col="condition",
                group_order=cortex_order,
                hue_order=condition_order,
                group_to_test=group,
                title=f"{column} - Barplot ({group})",
                custom_palette=None,
                export_path=export_path,
                pairs=pairs
            )

In [ ]:
def export_dataframe_to_mat_struct(df, export_path=None):
    """
    Export a pandas DataFrame to a MATLAB-compatible .mat file as a struct.
    The filename is automatically built from the DataFrame's variable name
    and its 'foi' column (if present).

    Each DataFrame column becomes a field in the MATLAB struct.
    The .mat file can be loaded in MATLAB or Scilab using 'load(filename)'.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to export.

    export_path : str, optional
        Directory to save the file. Defaults to the current directory.

    Notes
    -----
    - Filename is auto-generated as "<df_name>_<foi>.mat" if 'foi' exists,
      otherwise just "<df_name>.mat".
    - Numeric columns are saved as numeric arrays.
    - String or categorical columns are saved as cell arrays of strings.
    - Fully compatible with MATLAB and Scilab.
    """

    # --- Try to infer the variable name of df ---
    df_name = None
    for name, val in inspect.currentframe().f_back.f_locals.items():
        if val is df:
            df_name = name
            break
    if df_name is None:
        df_name = "dataframe"

    # --- Build filename based on foi if available ---
    if "foi" in df.columns:
        try:
            foi_val = str(df["foi"].iloc[0]).replace('.', 'p')
            filename = f"{df_name}_{foi_val}"
        except Exception:
            filename = df_name
    else:
        filename = df_name

    # --- Ensure export directory exists ---
    if export_path is None:
        export_path = os.getcwd()
    os.makedirs(export_path, exist_ok=True)

    # --- Convert DataFrame columns to MATLAB-compatible dictionary ---
    data_dict = {}
    for col in df.columns:
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
            data_dict[col] = np.array(df[col].astype(str).to_list(), dtype=object)
        else:
            data_dict[col] = np.array(df[col].to_numpy())

    # --- Save to .mat file ---
    save_path = os.path.join(export_path, f"{filename}.mat")
    savemat(save_path, {df_name: data_dict})

    print(f"Exported '{df_name}' to MATLAB struct at {save_path}")


## Pipelines

In [ ]:
def single_file_pipeline(file_path):
    """
    Runs the full processing pipeline on FOOOF and burst data extracted by superlet_burst_detection
    - Loads signal and metadata

    Parameters:
        config (dict): Configuration dictionary with all analysis parameters.
        file_path : path of the neo .mat file to analyze

    Returns:
        None (results are saved to disk as plots and CSV/pickle files)
    """

    # --- Retrieve file informations ---
    animal_id, group, condition, cortex, file_name = parse_file_properties(file_path)

    # --- Load epochs data (JSON file that stores dropped epochs info) ---
    drop_save_path = file_name + "_dropped_epochs.json"

    # Make sure the JSON file exists
    import json, os
    if not os.path.exists(drop_save_path):
        print(f"Warning: JSON file not found at {drop_save_path}")
        kept_indices = []
        total_epochs_before = None
        total_epochs_after = None
    else:
        with open(drop_save_path, "r") as f:
            dropped_info = json.load(f)
        kept_indices = dropped_info.get("kept_indices", [])
        total_epochs_before = dropped_info.get("total_epochs_before", None)
        total_epochs_after = dropped_info.get("total_epochs_after", None)

    # --- Gather file informations in a same file ---
    file_info = {
        "animal_id": animal_id,
        "group": group,
        "condition": condition,
        "cortex": cortex,
        "file_name": file_name,

        # Parameters from superlet_burst_detection
        "epoch_duration": 9.099,
        "ch_to_analyse": "CH28",
        "freq_to_analyse": "personalized",

        "Freq_Bands": {          # Frequency bands used for filtering and plotting
            "delta": [1.0, 4.0],
            "theta": [4.0, 8.0],
            "alpha": [8.0, 13.0],
            "beta": [13.0, 30.0],
            "low_gamma": [30.0, 50.0],
            "high_gamma": [50.0, 80.0],
            "gamma": [30.0, 80.0],
            "personalized": [1.0, 100.0],
        },

        "foi": "low_gamma",          # Frequency band to focus on
        "min_cycle": None,          # Minimum number of oscillatory cycles for a burst

        # Information loaded from the dropped_epochs JSON
        "epoch_idx": kept_indices,
        "initial_n_epoch": total_epochs_before,
        "n_epoch": total_epochs_after,
    }

    #Retrieve analyzed data for both components
    single_burst_results, burst_results = burst_results_analysis(file_info)
    fooof_results = fooof_results_analysis(file_info)
    
    return single_burst_results, burst_results, fooof_results


    

In [ ]:
def all_files_pipeline (file_paths):
    # ----------------------------------------------------------------------
    # Part 1: Retreive data and calculations obtained from each animal
    # ----------------------------------------------------------------------
    # Initiate list to collect per-animal results
    all_burst_results = []
    all_fooof_results = []

    for file_path in file_paths:
        #Calculate per-animal results
        burst_results, fooof_results = single_file_pipeline(file_path)

        # Combine burst and fooof results
        all_burst_results.append(burst_results)
        all_fooof_results.append(fooof_results)
    
    # Convert results to DataFrame
    df_burst = pd.DataFrame(all_burst_results)
    df_fooof = pd.DataFrame(all_fooof_results)


    # ----------------------------------------------------------------------
    # Part 2: Plotting graphs for burst data
    # ----------------------------------------------------------------------

    #Define order we want the bars to show
    cortex_order = ["Contra", "Peri"], 
    condition_order = ["Pre", "VNS", "Post"]
    group_order = ["sVNS", "VNS"]

    pairs = [
        (("Contra", "Pre"), ("Contra", "VNS")),
        (("Peri", "Pre"), ("Peri", "VNS"))
    ]

    plot_bar(
        df_burst,
        value_col="mean_peak_freq",
        group_col="cortex",
        hue_col="condition",
        group_order=["Contra", "Peri"],
        hue_order=["Pre", "VNS", "Post"],
        title="Mean Peak Frequency with Significance",
        pairs=pairs,
        test='t-test_ind'
    )

### 2) Run the code

In [ ]:
# Find the files we want to analyze
file_paths = file_choice()

In [ ]:
# ----------------------------------------------------------------------
# Part 1: Retreive data and calculations obtained from each animal
# ----------------------------------------------------------------------

# Initialize containers
all_single_burst_results = []   # all individual bursts (for LMM/GLMM)
all_burst_results = []          # per-animal summary metrics
all_fooof_results = []          # per-animal fooof metrics

# Loop through all files
for file_path in file_paths:
    # Run your per-file analysis pipeline
    single_burst_results, burst_results, fooof_results = single_file_pipeline(file_path)

    # Store per-animal summaries
    all_burst_results.append(burst_results)
    all_fooof_results.append(fooof_results)

    # Store individual burst-level results
    all_single_burst_results.append(single_burst_results)

# Convert lists of dictionaries to DataFrames
df_burst = pd.DataFrame(all_burst_results)
df_fooof = pd.DataFrame(all_fooof_results)

# Concatenate all single-burst DataFrames into one big dataset
df_single_bursts = pd.concat(all_single_burst_results, ignore_index=True)


# ----------------------------------------------------------------------
# Part 2: Plotting graphs for burst and fooof data
# ----------------------------------------------------------------------
# Make array of column names
burst_columns_single = ["mean_n_burst", "mean_peak_freq","mean_peak_amp_iter","mean_peak_amp_base","mean_peak_time","mean_fwhm_freq","mean_fwhm_time","mean_gaus_vol_model"]
burst_columns_array = ["epoch_mean_peak_freq", "epoch_mean_peak_amp_iter", "epoch_mean_peak_amp_base", "epoch_mean_peak_time", "epoch_mean_fwhm_freq", "epoch_mean_fwhm_time","epoch_mean_gaus_vol_model"]

fooof_columns_single = ["mean_offset", "mean_knee", "mean_exponent"]
fooof_columns_array = ["epoch_offset","epoch_knee","epoch_exponent"]

#Define the export folder 
export_path = r"./results"
os.makedirs(export_path, exist_ok=True)

# Plot results
plot_several_columns(df_burst,burst_columns_single,burst_columns_array,export_path)
plot_several_columns(df_fooof,fooof_columns_single,fooof_columns_array,export_path)

#Export tables into matlab structs
export_dataframe_to_mat_struct(df_burst, export_path)
export_dataframe_to_mat_struct(df_fooof, export_path)

In [ ]:
burst_columns_single_burst = ["peak_freq","peak_amp_iter","peak_amp_base","peak_time","fwhm_freq","fwhm_time","gaus_vol_model"]
plot_several_columns_all_burst(
    df_data=df_single_bursts,
    columns_to_analyze = burst_columns_single_burst,
    export_path = export_path
)